<a href="https://colab.research.google.com/github/bomgom02-netizen/skills_claude/blob/main/Sell_BITU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import yfinance as yf
import requests
import time

# [설정] 텔레그램 정보
TELEGRAM_TOKEN = '7796181604:AAGo0xhEHPIR6aza8vsMVAVEINdI1_4f08k'
CHAT_ID = '8528061505'

# [매도 전략 설정]
BITU_SELL_TARGET = 13.90  # 손절 목표가 상단
BITU_TICKER = "BITU"

def send_telegram(message):
    url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage"
    params = {'chat_id': CHAT_ID, 'text': message}
    try:
        requests.get(url, params=params)
    except Exception as e:
        print(f"❌ 전송 오류: {e}")

def monitor_bitu_exit():
    print(f"🛡️ {BITU_TICKER} 리스크 관리 모드 가동 중...")
    alert_sent_stop_loss = False # 손절 알림 반복 방지
    alert_sent_partial_sell_1 = False # 1차 분할매도 알림 반복 방지
    alert_sent_partial_sell_2 = False # 2차 분할매도 알림 반복 방지

    while True:
        try:
            # 실시간 데이터 수집
            ticker = yf.Ticker(BITU_TICKER)
            price = ticker.history(period='1d')['Close'].iloc[-1]

            print(f"🕒 현재가: ${price:.2f} | 손절 목표가: ${BITU_SELL_TARGET:.2f}")

            # 매도 시그널 로직 (손절)
            if price <= BITU_SELL_TARGET and not alert_sent_stop_loss:
                msg = (f"🚨 [BITU 매도 실행 알림]\n"
                       f"🛡️ 현재가: ${price:.2f}\n"
                       f"📉 설정한 손절가(${BITU_SELL_SELL_TARGET}) 이하 도달!\n"
                       f"💡 즉시 매도 후 현금화하여 주도주 재투자 자금으로 확보하십시오.")
                send_telegram(msg)
                print(msg)
                alert_sent_stop_loss = True # 한 번 알림 후 중단
            elif price > BITU_SELL_TARGET: # 가격이 다시 회복되면 손절 알림 초기화
                alert_sent_stop_loss = False

            # 분할 매도 로직
            # 1차 분할 매도
            if price >= 15.00 and price < 16.00 and not alert_sent_partial_sell_1:
                msg = f"🔔 [BITU 분할 매도 시그널 - 1차]\n현재가 ${price:.2f} 도달. 보유 물량의 30% 매도를 권고합니다."
                send_telegram(msg)
                print(msg)
                alert_sent_partial_sell_1 = True
            elif price < 15.00: # 가격이 다시 1차 분할 매도 지점 아래로 내려가면 알림 초기화
                alert_sent_partial_sell_1 = False

            # 2차 분할 매도
            if price >= 16.00 and not alert_sent_partial_sell_2:
                msg = f"🔔 [BITU 분할 매도 시그널 - 2차]\n현재가 ${price:.2f} 도달. 보유 물량의 40% 추가 매도를 권고합니다."
                send_telegram(msg)
                print(msg)
                alert_sent_partial_sell_2 = True
            elif price < 16.00: # 가격이 다시 2차 분할 매도 지점 아래로 내려가면 알림 초기화
                alert_sent_partial_sell_2 = False

        except Exception as e:
            print(f"❌ 데이터 수집 오류: {e}")

        time.sleep(60) # 1분 단위 감시

if __name__ == "__main__":
    monitor_bitu_exit()

In [ ]:
import yfinance as yf
import requests
import time
from datetime import datetime

# ==========================================
# 🛡️ [설정] 텔레그램 및 포트폴리오 변수
# ==========================================
TELEGRAM_CONFIG = {
    "TOKEN": "7796181604:AAGo0xhEHPIR6aza8vsMVAVEINdI1_4f08k",
    "CHAT_ID": "8528061505"
}

# 감시 대상 종목 설정 (ticker, 목표가 등)
TARGETS = {
    'ARM': {'ticker': 'ARM', 'sell_price': 155.0},
    'BITU': {'ticker': 'BITU'},
    '삼성전자': {'ticker': '005930.KS', 'buy_price': 190000},
    '한화에어로스페이스': {'ticker': '012450.KS', 'buy_price': 1400000},
    '고려아연': {'ticker': '010130.KS', 'buy_price': 500000}
}

# ==========================================
# 🔧 [핵심 함수]
# ==========================================

def send_telegram(message):
    url = f"https://api.telegram.org/bot{TELEGRAM_CONFIG['TOKEN']}/sendMessage"
    params = {'chat_id': TELEGRAM_CONFIG['CHAT_ID'], 'text': message}
    try:
        requests.get(url, params=params)
    except Exception as e:
        print(f"❌ 전송 오류: {e}")

def get_bitu_signal(price):
    """BITU 분할 매도 로직"""
    if price >= 16.00:
        return "🚨 [BITU 2차 분할 매도] 목표가 $16.00 도달! 보유량 40% 매도 권고."
    elif price >= 15.00:
        return "🔔 [BITU 1차 분할 매도] 목표가 $15.00 도달! 보유량 30% 매도 권고."
    elif price <= 13.90:
        return "⚠️ [BITU 손절] 손절가 $13.90 이하! 리스크 차단 필요."
    return None

# ==========================================
# 🚀 [메인 루프]
# ==========================================

def watch_system():
    print("🚀 리스크 관리 통합 시스템 가동 중...")
    while True:
        try:
            # 1. 데이터 수집
            for name, info in TARGETS.items():
                df = yf.Ticker(info['ticker']).history(period='1d')
                if df.empty: continue

                price = df['Close'].iloc[-1]

                # 2. BITU 특별 로직 처리
                if name == 'BITU':
                    signal = get_bitu_signal(price)
                    if signal:
                        send_telegram(f"🔔 [BITU 관리] 현재가: ${price:.2f}\n{signal}")

                # 3. 일반 매수/매도 로직
                elif 'buy_price' in info and price <= info['buy_price']:
                    send_telegram(f"📢 [매수 시그널] {name}: {price:,.0f}원 (목표가 이하)")
                elif 'sell_price' in info and price <= info['sell_price']:
                    send_telegram(f"🚨 [매도 시그널] {name}: {price:,.2f} (손절가 이하)")

        except Exception as e:
            print(f"❌ 데이터 조회 오류: {e}")

        time.sleep(300) # 5분 간격 모니터링

if __name__ == "__main__":
    watch_system()

🚀 리스크 관리 통합 시스템 가동 중...
